In [ ]:
# ── INSPECTION ONLY — READ-ONLY — nothing is written or moved ──────────────────
import os, json
from pathlib import Path

# 1. Runtime info
import platform, subprocess
print('=== RUNTIME ===')
print(f'Platform : {platform.platform()}')
print(f'Python   : {platform.python_version()}')

# 2. GPU
try:
    gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True).strip()
    print(f'GPU      : {gpu}')
except Exception as e:
    print(f'GPU      : Not available ({e})')

# 3. Drive mount status
drive_root = Path('/content/drive')
if drive_root.exists() and list(drive_root.iterdir()):
    print(f'\nDrive    : MOUNTED at {drive_root}')
    for p in sorted(drive_root.iterdir()):
        print(f'  {p.name}/')
else:
    print('\nDrive    : NOT YET MOUNTED')

# 4. Scan for FetalBrain_XAI_Workspace anywhere under /content/drive
print('\n=== SEARCHING FOR FetalBrain_XAI_Workspace ===')
workspace_candidates = list(Path('/content/drive').rglob('FetalBrain_XAI_Workspace')) if drive_root.exists() else []
if workspace_candidates:
    for ws in workspace_candidates:
        print(f'  FOUND: {ws}')
        # List top-level contents
        for child in sorted(ws.iterdir()):
            size_str = f'{child.stat().st_size:,} bytes' if child.is_file() else 'DIR'
            print(f'    {child.name:<40} {size_str}')
else:
    print('  Not found (Drive may not be mounted yet)')
    # Also check /content directly
    content = Path('/content')
    print('\n/content contents:')
    for p in sorted(content.iterdir()):
        print(f'  {p.name}')

In [ ]:
# ── Mount Google Drive — requires YOUR browser authorization popup ───────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('Drive mounted successfully.')

In [ ]:
# ── POST-MOUNT INSPECTION — READ-ONLY ─────────────────────────────────────
# Run AFTER you authorize Drive above.
# Reports the complete structure of FetalBrain_XAI_Workspace.
# Nothing is written, renamed, moved, or deleted.
import os, json
from pathlib import Path

def tree(p: Path, indent: int = 0, max_depth: int = 4, max_children: int = 30):
    if indent > max_depth:
        return
    children = sorted(p.iterdir()) if p.is_dir() else []
    shown = children[:max_children]
    for child in shown:
        prefix = '  ' * indent
        if child.is_dir():
            n = len(list(child.iterdir()))
            print(f'{prefix}📁 {child.name}/  ({n} items)')
            tree(child, indent + 1, max_depth, max_children)
        else:
            size = child.stat().st_size
            size_str = f'{size/1024:.1f} KB' if size < 1e6 else f'{size/1e6:.1f} MB'
            print(f'{prefix}📄 {child.name}  [{size_str}]')
    if len(children) > max_children:
        print(f'{'  ' * indent}... and {len(children) - max_children} more')

ws_search = list(Path('/content/drive').rglob('FetalBrain_XAI_Workspace'))
if ws_search:
    ws = ws_search[0]
    print(f'Workspace found: {ws}\n')
    tree(ws)
else:
    print('FetalBrain_XAI_Workspace not found. Listing all of MyDrive:')
    tree(Path('/content/drive/MyDrive'), max_depth=2)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 1 — Mount Drive
# YOU must click the Google auth popup that appears below.
# Antigravity cannot do this step for you.
# ════════════════════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('✅ Drive mounted.')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 2 — Full workspace + source code inspection (READ-ONLY)
# Run AFTER Drive is mounted. Nothing is modified.
# ════════════════════════════════════════════════════════════════════════════
from pathlib import Path
import os

ROOT = Path('/content/drive/MyDrive/FetalBrain_XAI_Workspace')
FOLDERS = [
    '00_Documentation', '01_Source_Code', '02_Original_Dataset',
    '03_Manifests_And_Labels', '04_Current_Results', '05_Model_Backups',
    '06_External_Data_Raw', '07_External_Data_Staging', '08_Antigravity_Reports'
]

print(f'Workspace root : {ROOT}')
print(f'Exists         : {ROOT.exists()}\n')

if not ROOT.exists():
    raise RuntimeError('Workspace not found — mount Drive first')

for folder in FOLDERS:
    p = ROOT / folder
    if not p.exists():
        print(f'  MISSING  {folder}')
        continue
    all_files = [f for f in p.rglob('*') if f.is_file()]
    total_mb = sum(f.stat().st_size for f in all_files) / 1e6
    print(f'\n► {folder}  ({len(all_files)} files, {total_mb:.2f} MB)')
    # Show every file so Antigravity can see exactly what exists
    for f in sorted(all_files)[:50]:
        rel = f.relative_to(p)
        size_kb = f.stat().st_size / 1024
        print(f'    {str(rel):<60} {size_kb:>8.1f} KB')
    if len(all_files) > 50:
        print(f'    ... and {len(all_files)-50} more files')

print('\n✔ Inspection complete — no files were modified.')

In [ ]:
# READ ALL SOURCE FILES — so Antigravity can analyze them
from pathlib import Path

SRC = Path('/content/drive/MyDrive/FetalBrain_XAI_Workspace/01_Source_Code')
files = sorted(SRC.glob('*.py'))

for f in files:
    print(f'\n{"="*80}')
    print(f'FILE: {f.name}  ({f.stat().st_size/1024:.1f} KB)')
    print('='*80)
    print(f.read_text(errors='replace'))
    print(f'\n--- END {f.name} ---')


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL A — FIX FILENAMES & COPY SCRIPTS TO RUNTIME
# Renames the (1) duplicates, removes double-extension, copies all to /content/src
# ════════════════════════════════════════════════════════════════════════════════
import shutil
from pathlib import Path

SRC_DRIVE = Path('/content/drive/MyDrive/FetalBrain_XAI_Workspace/01_Source_Code')
SRC_RT    = Path('/content/src')
SRC_RT.mkdir(exist_ok=True)

# Canonical name mapping: drive filename -> clean runtime name
FILE_MAP = {
    '01_dataset_audit.py':       '01_dataset_audit.py',
    '02_rebuild_dataset.py':     '02_rebuild_dataset.py',
    '03_train.py':               '03_train.py',
    '04_evaluate (1).py':        '04_evaluate.py',     # fix (1) and space
    '05_diagnostics (1).py':     '05_diagnostics.py',  # fix (1) and space
    '06_insights (1).py':        '06_insights.py',     # fix (1) and space
    '07_build_grouped.py':       '07_build_grouped.py',
    '08_compare_runs.py':        '08_compare_runs.py', # canonical (not the (1) copy)
    '09_compare_abnormal.py':    '09_compare_abnormal.py',
    '12_feature_probe.py':       '12_feature_probe.py',
    'Foetal_patched.py':         'Foetal_patched.py',  # patched is the good one
    # Skip: '08_compare_runs (1).py' -> duplicate of 08_compare_runs.py
    # Skip: 'Foetal.py.py'           -> double extension, use patched instead
}

print('Copying scripts to runtime...')
for drive_name, rt_name in FILE_MAP.items():
    src = SRC_DRIVE / drive_name
    dst = SRC_RT / rt_name
    if src.exists():
        shutil.copy2(src, dst)
        print(f'  OK  {drive_name:45s} -> {rt_name}')
    else:
        print(f'  !!  MISSING: {drive_name}')

# Also copy the class map JSON
cmap_src = Path('/content/drive/MyDrive/FetalBrain_XAI_Workspace/03_Manifests_And_Labels/class_map_grouped.json')
cmap_dst = SRC_RT / 'class_map_grouped.json'
if cmap_src.exists():
    shutil.copy2(cmap_src, cmap_dst)
    print(f'  OK  class_map_grouped.json')

print(f'\nRuntime source dir: {SRC_RT}')
print('Files ready:', [f.name for f in sorted(SRC_RT.iterdir())])


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL B — INSTALL DEPENDENCIES
# ════════════════════════════════════════════════════════════════════════════════
import subprocess, sys

packages = [
    'opencv-python-headless',  # cv2 for CLAHE (headless = no display needed)
    'scikit-learn',
    'Pillow',
    # tensorflow and numpy are already available in Colab
]

print('Installing required packages...')
for pkg in packages:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', pkg],
        capture_output=True, text=True
    )
    status = '✅' if result.returncode == 0 else '❌'
    print(f'  {status} {pkg}')
    if result.returncode != 0:
        print(result.stderr[:300])

# Verify key imports
import tensorflow as tf
import cv2
import sklearn
from PIL import Image

print(f'\nTensorFlow : {tf.__version__}')
print(f'OpenCV     : {cv2.__version__}')
print(f'sklearn    : {sklearn.__version__}')

gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs       : {gpus if gpus else "NONE — go to Runtime > Change runtime type > T4 GPU"}')


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL C — EXTRACT DATASET
# Unzips the 57 MB dataset from Drive to /content/ds/ in the runtime.
# Nothing is written back to Drive.
# ════════════════════════════════════════════════════════════════════════════════
import zipfile
from pathlib import Path

ZIP_PATH = Path('/content/drive/MyDrive/FetalBrain_XAI_Workspace/02_Original_Dataset/Fetal Brain Abnormalities Ultrasound.v1i.multiclass.zip')
DS_OUT   = Path('/content/ds')

if DS_OUT.exists() and any(DS_OUT.iterdir()):
    print(f'Dataset already extracted at {DS_OUT}')
else:
    print(f'Extracting {ZIP_PATH.name}  ({ZIP_PATH.stat().st_size/1e6:.1f} MB)...')
    DS_OUT.mkdir(exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DS_OUT)
    print('Done.')

# Show what we got
print(f'\nContents of {DS_OUT}:')
for p in sorted(DS_OUT.iterdir()):
    if p.is_dir():
        n = sum(1 for _ in p.rglob('*') if _.is_file())
        print(f'  {p.name}/  ({n} files)')
    else:
        print(f'  {p.name}  ({p.stat().st_size/1024:.1f} KB)')


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL D — STEP 1: DATASET AUDIT (read-only)
# ════════════════════════════════════════════════════════════════════════════════
import subprocess, sys
from pathlib import Path

SRC = '/content/src'

# Find the actual extracted dataset root (handles nested zip layouts)
DS = Path('/content/ds')
possible_roots = [DS] + list(DS.iterdir()) if DS.exists() else []
ds_root = None
for p in possible_roots:
    if p.is_dir() and any((p / s).is_dir() for s in ('train', 'valid', 'test')):
        ds_root = p
        break

if ds_root is None:
    raise RuntimeError(f'Could not find train/valid/test splits under {DS}. Check the zip contents.')
print(f'Dataset root found: {ds_root}')

result = subprocess.run(
    [sys.executable, f'{SRC}/01_dataset_audit.py',
     '--src', str(ds_root),
     '--out', '/content/audit'],
    capture_output=False, text=True
)
print('\nReturn code:', result.returncode)


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL E — STEP 2: REBUILD DATASET (case-level leak-free split)
# ════════════════════════════════════════════════════════════════════════════════
import subprocess, sys, os
from pathlib import Path

SRC = '/content/src'
ds_root = str(next(
    p for p in [Path('/content/ds')] + list(Path('/content/ds').iterdir())
    if p.is_dir() and any((p/s).is_dir() for s in ('train','valid','test'))
))

os.chdir(SRC)  # 02 uses importlib to load 01 from same dir

result = subprocess.run(
    [sys.executable, f'{SRC}/02_rebuild_dataset.py',
     '--src', ds_root,
     '--out', '/content/dataset_rebuilt',
     '--protocol', 'group',
     '--aug-policy', 'train-only',
     '--min-cases', '3',
     '--small-class', 'train-only'],
    capture_output=False, text=True,
    cwd=SRC  # ensures importlib can find 01_dataset_audit.py
)
print('\nReturn code:', result.returncode)


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL F — STEP 3: TRAIN XCEPTION
# ⚠️ REQUIRES GPU RUNTIME. Go to Runtime > Change runtime type > T4 GPU first.
# This runs two phases: frozen head warmup (12 epochs) + fine-tuning (up to 60 epochs).
# EarlyStopping will stop earlier if val_macro_f1 plateaus.
# ════════════════════════════════════════════════════════════════════════════════
import subprocess, sys, tensorflow as tf

# Check GPU
gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    print('⚠️  No GPU detected! Training will be very slow on CPU.')
    print('   Go to Runtime > Change runtime type > T4 GPU, then re-run from Cell A.')
else:
    print(f'✅ GPU: {gpus[0].name}')
    result = subprocess.run(
        [sys.executable, '/content/src/03_train.py',
         '--data', '/content/dataset_rebuilt',
         '--model', 'xception',
         '--out', '/content/models',
         '--batch-size', '16',
         '--warmup-epochs', '12',
         '--epochs', '60',
         '--ft-lr', '1e-5',
         '--unfreeze-from', 'block12',
         '--patience', '10',
         '--mixed-precision'],
        capture_output=False, text=True
    )
    print('\nReturn code:', result.returncode)


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL G — STEP 4: EVALUATION (with Test-Time Augmentation & Confusion Matrices)
# Evaluates trained Xception on held-out leak-free test split.
# ════════════════════════════════════════════════════════════════════════════════
import subprocess, sys
from pathlib import Path

SRC = '/content/src'
DATA = '/content/dataset_rebuilt'
MODELS = '/content/models'
OUT = '/content/results'

Path(OUT).mkdir(parents=True, exist_ok=True)

result = subprocess.run(
    [sys.executable, f'{SRC}/04_evaluate.py',
     '--data', DATA,
     '--models', MODELS,
     '--out', OUT,
     '--tta'],
    capture_output=False, text=True
)
print('\nEvaluation return code:', result.returncode)

# Display summary files if generated
res_table = Path(OUT) / 'results_table.md'
if res_table.exists():
    print('\n' + '='*70)
    print(res_table.read_text())


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL H — STEP 5: DIAGNOSTICS (Normal vs Abnormal confound & Ventriculomegaly)
# Answers: 1) Normal-vs-abnormal acquisition shortcut
#          2) Confound-free abnormality discrimination
#          3) Ventriculomegaly severity merge analysis
# ════════════════════════════════════════════════════════════════════════════════
import subprocess, sys
from pathlib import Path

SRC = '/content/src'
DATA = '/content/dataset_rebuilt'
MODELS = '/content/models'
OUT = '/content/results'

result = subprocess.run(
    [sys.executable, f'{SRC}/05_diagnostics.py',
     '--data', DATA,
     '--models', MODELS,
     '--model', 'xception',
     '--out', OUT],
    capture_output=False, text=True
)
print('\nDiagnostics return code:', result.returncode)

diag_txt = Path(OUT) / 'diagnostics.txt'
if diag_txt.exists():
    print('\n' + '='*70)
    print(diag_txt.read_text())


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL I — STEP 6: INSIGHTS (Pixel Shortcut Test, Overfitting, Preprocessing Audit)
# ════════════════════════════════════════════════════════════════════════════════
import subprocess, sys
from pathlib import Path

SRC = '/content/src'
DATA = '/content/dataset_rebuilt'
MODELS = '/content/models'
OUT = '/content/results'

result = subprocess.run(
    [sys.executable, f'{SRC}/06_insights.py',
     '--data', DATA,
     '--models', MODELS,
     '--model', 'xception',
     '--out', OUT],
    capture_output=False, text=True
)
print('\nInsights return code:', result.returncode)

ins_txt = Path(OUT) / 'insights.txt'
if ins_txt.exists():
    print('\n' + '='*70)
    print(ins_txt.read_text())


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL J — STEP 7: BUILD DERIVED EXPERIMENT DATASETS (Grouped & Abnormal-Only)
# Inherits exact case-level partitions from dataset_rebuilt (leak-free guaranteed).
# ════════════════════════════════════════════════════════════════════════════════
import subprocess, sys
from pathlib import Path

SRC = '/content/src'
DATA = '/content/dataset_rebuilt'
CMAP = f'{SRC}/class_map_grouped.json'

# 1. Build Grouped Families dataset
print('Building Grouped dataset...')
res1 = subprocess.run(
    [sys.executable, f'{SRC}/07_build_grouped.py',
     '--like', DATA,
     '--class-map', CMAP,
     '--out', '/content/dataset_grouped'],
    capture_output=False, text=True
)

# 2. Build Abnormal-Only dataset (normal class removed)
print('\nBuilding Abnormal-Only dataset...')
res2 = subprocess.run(
    [sys.executable, f'{SRC}/07_build_grouped.py',
     '--like', DATA,
     '--drop-classes', 'normal',
     '--out', '/content/dataset_abnormal'],
    capture_output=False, text=True
)

print('\nGrouped return code :', res1.returncode)
print('Abnormal return code:', res2.returncode)


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL K — STEP 8: TRAIN ABNORMAL-ONLY MODEL (EXPERIMENT 3)
# Trains Xception specifically on abnormal conditions to eliminate normal-bias.
# ════════════════════════════════════════════════════════════════════════════════
import subprocess, sys, tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    print('⚠️ GPU recommended for training.')
else:
    print(f'✅ GPU: {gpus[0].name}')

result = subprocess.run(
    [sys.executable, '/content/src/03_train.py',
     '--data', '/content/dataset_abnormal',
     '--model', 'xception',
     '--out', '/content/models_abnormal',
     '--batch-size', '16',
     '--warmup-epochs', '12',
     '--epochs', '60',
     '--ft-lr', '1e-5',
     '--unfreeze-from', 'block12',
     '--patience', '10',
     '--mixed-precision'],
    capture_output=False, text=True
)
print('\nAbnormal-only training return code:', result.returncode)


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL L — STEP 9: EXPERIMENT 3 HEAD-TO-HEAD COMPARISON
# Evaluates 16-class restricted vs Abnormal-only on identical abnormal test cases.
# Runs McNemar's exact test.
# ════════════════════════════════════════════════════════════════════════════════
import subprocess, sys
from pathlib import Path

SRC = '/content/src'
DATA_AB = '/content/dataset_abnormal'
M16 = '/content/models/best_model_xception.keras'
M_AB = '/content/models_abnormal/best_model_xception.keras'
OUT = '/content/results_abnormal'

result = subprocess.run(
    [sys.executable, f'{SRC}/09_compare_abnormal.py',
     '--data-abnormal', DATA_AB,
     '--model-16', M16,
     '--model-abnormal', M_AB,
     '--out', OUT],
    capture_output=False, text=True
)
print('\nComparison return code:', result.returncode)

comp_txt = Path(OUT) / 'comparison_vs_16class.txt'
if comp_txt.exists():
    print('\n' + '='*70)
    print(comp_txt.read_text())


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL M — STEP 10: L1 FEATURE PROBE & SELECTION
# Extracts frozen features, sweeps L1 strength C, selects sparse deep features.
# ════════════════════════════════════════════════════════════════════════════════
import subprocess, sys
from pathlib import Path

SRC = '/content/src'
DATA = '/content/dataset_rebuilt'
MODELS = '/content/models'
OUT = '/content/results_probe'

result = subprocess.run(
    [sys.executable, f'{SRC}/12_feature_probe.py',
     '--data', DATA,
     '--models', MODELS,
     '--model', 'xception',
     '--backbone', 'finetuned',
     '--out', OUT],
    capture_output=False, text=True
)
print('\nFeature probe return code:', result.returncode)

probe_txt = Path(OUT) / 'probe_finetuned.txt'
if probe_txt.exists():
    print('\n' + '='*70)
    print(probe_txt.read_text())


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL N — STEP 11: ARCHIVE & SYNC ALL RESULTS BACK TO GOOGLE DRIVE
# Safely writes timestamped reports and results back to Drive:
#   1. /content/drive/MyDrive/FetalBrain_XAI_Workspace/08_Antigravity_Reports/{timestamp}/
#   2. /content/drive/MyDrive/FetalBrain_XAI_Workspace/04_Current_Results/
# ════════════════════════════════════════════════════════════════════════════════
import shutil, datetime
from pathlib import Path

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
DRIVE_WORKSPACE = Path('/content/drive/MyDrive/FetalBrain_XAI_Workspace')
REPORT_DIR = DRIVE_WORKSPACE / '08_Antigravity_Reports' / f'run_{timestamp}'
RESULTS_DIR = DRIVE_WORKSPACE / '04_Current_Results'

REPORT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sync_folders = [
    ('/content/audit', 'audit'),
    ('/content/results', 'results_main'),
    ('/content/results_abnormal', 'results_abnormal'),
    ('/content/results_probe', 'results_probe'),
]

print(f'Archiving session artifacts to Drive:\n  -> {REPORT_DIR}')
for src_str, name in sync_folders:
    src_p = Path(src_str)
    if src_p.exists():
        dst_report = REPORT_DIR / name
        dst_current = RESULTS_DIR / name
        # Copy to timestamped report
        shutil.copytree(src_p, dst_report, dirs_exist_ok=True)
        # Copy to current results
        shutil.copytree(src_p, dst_current, dirs_exist_ok=True)
        print(f'  ✅ Synced {src_str} ({len(list(src_p.iterdir()))} items)')
    else:
        print(f'  ⚠️ Skipped {src_str} (not generated in this run)')

# Also backup trained models
MODELS_SRC = Path('/content/models')
MODELS_DST = DRIVE_WORKSPACE / '05_Model_Backups' / f'models_{timestamp}'
if MODELS_SRC.exists() and any(MODELS_SRC.iterdir()):
    MODELS_DST.mkdir(parents=True, exist_ok=True)
    shutil.copytree(MODELS_SRC, MODELS_DST, dirs_exist_ok=True)
    print(f'  ✅ Backed up models to {MODELS_DST}')

print('\n🎉 All deliverables synced to Google Drive successfully!')
